<a href="https://colab.research.google.com/github/dhiru69-tech/ML-Techniques-Lab-Project/blob/main/notebooks/day11_knn_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 11: K-Nearest Neighbours (KNN)
**Course:** BO CDA 205, Machine Learning Techniques
**Dataset:** German Credit (1000 customers, 21 columns)
**Syllabus link:** Supervised learning, nearest neighbour classifiers

> **Goal of today:** Build KNN by hand, see why scaling matters, choose k with cross-validation, observe the curse of dimensionality, and pick the decision threshold using the training data only.

 ## 1. Setup

Same workflow as before: stratified split, preprocessing fitted on the training set only, and a pipeline for cross-validation.

`make_pipe(prep, **params)` builds a pipeline with the preprocessor we pass in and a KNN model with the given settings.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.datasets import fetch_openml
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_validate,
                                     cross_val_score, cross_val_predict, GridSearchCV)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)

# Load data. bad = 1 is the positive class
data = fetch_openml(name="credit-g", version=1, as_frame=True)
df = data.frame
X = df.drop(columns=['class'])
y = df['class'].map({'good': 0, 'bad': 1}).astype(int)

# Stratified 80/20 split with a fixed seed
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()

# Preprocessing 1: numeric columns scaled, categorical columns one-hot encoded
preprocess = ColumnTransformer(
    [('num', StandardScaler(), num_cols),
     ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)],
    sparse_threshold=0
)
preprocess.set_output(transform="pandas")

# Preprocessing 2 (for the scaling experiment): numeric columns left unscaled
prep_raw = ColumnTransformer(
    [('num', 'passthrough', num_cols),
     ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)],
    sparse_threshold=0
)
prep_raw.set_output(transform="pandas")


def make_pipe(prep=preprocess, **params):
    """Pipeline = preprocessing + KNN. params go to the KNN model."""
    return Pipeline([('prep', clone(prep)), ('knn', KNeighborsClassifier(**params))])


def evaluate(name, y_true, pred):
    """Main metrics for the positive class (bad = 1), plus the Day 8 cost."""
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return {
        'model': name,
        'accuracy': round(accuracy_score(y_true, pred), 3),
        'precision': round(precision_score(y_true, pred, zero_division=0), 3),
        'recall': round(recall_score(y_true, pred, zero_division=0), 3),
        'f1': round(f1_score(y_true, pred, zero_division=0), 3),
        'FN': fn, 'FP': fp, 'cost': 5 * fn + fp,
    }


# 5-fold cross-validation that keeps the 70/30 ratio in every fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Processed (scaled + one-hot) arrays used for the by-hand parts
X_train_p = clone(preprocess).fit_transform(X_train)
X_test_p = clone(preprocess).fit(X_train).transform(X_test)
print("Train:", X_train_p.shape, "| Test:", X_test_p.shape)

Train: (800, 61) | Test: (200, 61)
